In [ ]:
# Linear Regression Data Analysis Project
# Energy Data Analysis Predicting Greenhouse Gas Emissions
# Includes an honest, leakage-free model and country-level validation,
# plus a full visual model-comparison suite.

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, GroupShuffleSplit
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error
from sklearn.preprocessing import StandardScaler

sns.set_style("whitegrid")
plt.rcParams["figure.dpi"] = 110


# ===== Section 1: Data Preparation & Preprocessing =====

print("="*70)
print("Predicting Greenhouse Gas Emissions Based on Energy Consumption")
print("="*70)

df = pd.read_csv("merged_energy_data.csv")
print(f"Dataset contains: {df.shape[0]:,} rows and {df.shape[1]} columns")


df.head()


missing = pd.DataFrame({
    "Count": df.isnull().sum(),
    "Percent": df.isnull().mean() * 100
})
missing = missing[missing["Count"] > 0].sort_values("Count", ascending=False)
print(f"Total missing values: {missing['Count'].sum():,}")
print(f"Columns with missing values: {len(missing)}")


selected_columns = [
    "country", "year",
    "greenhouse_gas_emissions",   # Target variable
    "coal_consumption", "gas_consumption", "oil_consumption",
    "fossil_fuel_consumption", "renewables_consumption", "nuclear_consumption",
    "hydro_consumption", "solar_consumption", "wind_consumption",
    "primary_energy_consumption", "population", "gdp", "energy_per_capita",
    "fossil_share_energy", "renewables_share_energy"
]
df_selected = df[selected_columns].copy()
print(f"Selected {len(selected_columns)} columns for analysis")


# Drop rows missing the target; drop very sparse rows; median-fill the rest
df_clean = df_selected.dropna(subset=["greenhouse_gas_emissions"])
min_non_nulls = int(0.8 * df_clean.shape[1])
df_clean = df_clean.dropna(thresh=min_non_nulls)
df_clean = df_clean.fillna(df_clean.median(numeric_only=True))

print(f"Clean dataset: {len(df_clean):,} rows")
print(f"Missing values remaining: {df_clean.isnull().sum().sum()}")


EPSILON = 0.001
df_clean["fossil_dependency_ratio"] = df_clean["fossil_fuel_consumption"] / (df_clean["primary_energy_consumption"] + EPSILON)
df_clean["renewable_ratio"] = df_clean["renewables_consumption"] / (df_clean["primary_energy_consumption"] + EPSILON)
df_clean["emissions_per_capita"] = df_clean["greenhouse_gas_emissions"] / (df_clean["population"] + EPSILON)
df_clean.reset_index(drop=True, inplace=True)

df_clean.to_excel("cleaned_energy_data.xlsx", index=False)
df_clean.to_csv("cleaned_energy_data.csv", index=False)
print("Saved cleaned dataset:", df_clean.shape)


# ===== Section 2: Data Exploration & Visualization =====

viz_cols = ["greenhouse_gas_emissions", "fossil_fuel_consumption",
            "renewables_consumption", "energy_per_capita", "population", "emissions_per_capita"]
df_clean[viz_cols].describe()


yearly_data = df_clean.groupby("year")[["greenhouse_gas_emissions", "fossil_fuel_consumption",
                                        "renewables_consumption", "population"]].sum()
fig, axes = plt.subplots(2, 2, figsize=(16, 10))
axes = axes.flatten()
ts_config = [
    ("greenhouse_gas_emissions", "red", "Global Emissions"),
    ("fossil_fuel_consumption", "brown", "Fossil Fuel Consumption"),
    ("renewables_consumption", "green", "Renewable Consumption"),
    ("population", "blue", "Global Population")
]
for i, (col, color, title) in enumerate(ts_config):
    axes[i].plot(yearly_data.index, yearly_data[col], marker="o", color=color, linewidth=2)
    axes[i].set_title(title, fontweight="bold")
    axes[i].set_xlabel("Year")
    axes[i].grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig("02_time_series.png", dpi=300)
plt.show()


# target distribution (raw vs log-transformed)
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
sns.histplot(df_clean["greenhouse_gas_emissions"], bins=40, color="crimson", ax=axes[0])
axes[0].set_title("Distribution: Greenhouse Gas Emissions (raw)", fontweight="bold")
sns.histplot(np.log1p(df_clean["greenhouse_gas_emissions"]), bins=40, color="teal", ax=axes[1])
axes[1].set_title("Distribution: log(1 + Emissions)", fontweight="bold")
plt.tight_layout()
plt.savefig("03_target_distribution.png", dpi=300)
plt.show()


# ===== Section 3: Correlation Analysis =====

corr_vars = [
    "greenhouse_gas_emissions", "coal_consumption", "gas_consumption", "oil_consumption",
    "fossil_fuel_consumption", "renewables_consumption", "nuclear_consumption",
    "primary_energy_consumption", "population", "gdp", "energy_per_capita",
    "fossil_share_energy", "renewables_share_energy", "fossil_dependency_ratio",
    "renewable_ratio", "emissions_per_capita"
]
corr_matrix = df_clean[corr_vars].corr()
target_corr = corr_matrix["greenhouse_gas_emissions"].sort_values(ascending=False).drop("greenhouse_gas_emissions")
target_corr


plt.figure(figsize=(16, 14))
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
sns.heatmap(corr_matrix, mask=mask, annot=True, fmt=".2f", cmap="RdYlGn",
            center=0, square=True, linewidths=0.5, cbar_kws={"shrink": 0.8})
plt.title("Correlation Heatmap", fontsize=16, fontweight="bold")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.savefig("04_correlation_heatmap.png", dpi=300)
plt.show()


# ranked correlation bar chart
plt.figure(figsize=(10, 7))
colors = ["crimson" if v > 0 else "steelblue" for v in target_corr.values]
sns.barplot(x=target_corr.values, y=target_corr.index, palette=colors)
plt.axvline(0, color="black", linewidth=0.8)
plt.title("Correlation with Greenhouse Gas Emissions, ranked", fontweight="bold")
plt.xlabel("Pearson correlation")
plt.tight_layout()
plt.savefig("05_correlation_ranked_bar.png", dpi=300)
plt.show()


# ===== Section 4: Model A - Original Baseline Model =====
# (coal_consumption, gas_consumption, oil_consumption, renewables_consumption,
# nuclear_consumption) are components of fossil_fuel_consumption or
# primary_energy_consumption, which are themselves near-identities of
# greenhouse_gas_emissions. Including them as predictors causes data leakage.
# This model is kept only as a baseline to show why it is misleading, and is
# followed by an honest, leakage-free model in Section 5.

target = "greenhouse_gas_emissions"
features_A = [
    "coal_consumption", "gas_consumption", "oil_consumption",
    "renewables_consumption", "nuclear_consumption",
    "population", "gdp", "energy_per_capita"
]
X_A, y_A = df_clean[features_A], df_clean[target]
X_train, X_test, y_train, y_test = train_test_split(X_A, y_A, test_size=0.2, random_state=42)

model_A = LinearRegression().fit(X_train, y_train)
pred_A_train = model_A.predict(X_train)
pred_A_test = model_A.predict(X_test)

def metrics(y_true, y_pred):
    return {
        "R2": r2_score(y_true, y_pred),
        "RMSE": np.sqrt(mean_squared_error(y_true, y_pred)),
        "MAE": mean_absolute_error(y_true, y_pred)
    }

mA_train, mA_test = metrics(y_train, pred_A_train), metrics(y_test, pred_A_test)
print("[Model A --inflated by leakage]")
print("Train:", mA_train)
print("Test :", mA_test)


# ===== Section 5: Model B - Honest Model (leakage-free features, country-level validation) =====

features_B = ["population", "gdp", "energy_per_capita", "fossil_dependency_ratio", "renewable_ratio"]
X_B, y_B = df_clean[features_B], df_clean[target]
groups = df_clean["country"]

gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
train_idx, test_idx = next(gss.split(X_B, y_B, groups))
X_train_B, X_test_B = X_B.iloc[train_idx], X_B.iloc[test_idx]
y_train_B, y_test_B = y_B.iloc[train_idx], y_B.iloc[test_idx]

model_B = LinearRegression().fit(X_train_B, y_train_B)
pred_B_train = model_B.predict(X_train_B)
pred_B_test = model_B.predict(X_test_B)

mB_train, mB_test = metrics(y_train_B, pred_B_train), metrics(y_test_B, pred_B_test)
print("[Model B --honest, country-held-out]")
print("Train:", mB_train)
print("Test :", mB_test)

coef_B = pd.DataFrame({"Feature": features_B, "Coeff": model_B.coef_}).sort_values("Coeff", key=abs, ascending=False)
print(coef_B.to_string(index=False))


# Interpretation: once the tautological fuel-consumption features are removed
# and the split respects country groupings, R2 drops from ~0.995 to a much
# lower than honest figure. This is the expected and correct result: GDP,
# population, and energy intensity alone are far weaker predictors of
# absolute emissions than fuel consumption is, because fuel consumption is
# emissions, almost by definition.


# ===== Section 6: Model Comparison =====

# 6.1 - Actual vs Predicted
fig, axes = plt.subplots(1, 2, figsize=(14, 5.5))
axes[0].scatter(y_test, pred_A_test, alpha=0.5, color="crimson")
lims = [y_test.min(), y_test.max()]
axes[0].plot(lims, lims, "k--", label="Perfect prediction")
axes[0].set_title(f"Model A (leaky features)\nTest R2={mA_test['R2']:.3f} | RMSE={mA_test['RMSE']:.1f}", fontweight="bold")
axes[0].set_xlabel("Actual"); axes[0].set_ylabel("Predicted"); axes[0].legend()

axes[1].scatter(y_test_B, pred_B_test, alpha=0.5, color="seagreen")
lims_b = [y_test_B.min(), y_test_B.max()]
axes[1].plot(lims_b, lims_b, "k--", label="Perfect prediction")
axes[1].set_title(f"Model B (honest, country-held-out)\nTest R2={mB_test['R2']:.3f} | RMSE={mB_test['RMSE']:.1f}", fontweight="bold")
axes[1].set_xlabel("Actual"); axes[1].set_ylabel("Predicted"); axes[1].legend()
plt.suptitle("Actual vs. Predicted Emissions: Model A vs Model B", fontsize=15, fontweight="bold")
plt.tight_layout()
plt.savefig("06_actual_vs_predicted.png", dpi=300)
plt.show()


# 6.2 - Residual plots (error patterns)
resid_A = y_test - pred_A_test
resid_B = y_test_B - pred_B_test

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].scatter(pred_A_test, resid_A, alpha=0.5, color="crimson")
axes[0].axhline(0, color="black", linestyle="--")
axes[0].set_title("Model A Residuals", fontweight="bold")
axes[0].set_xlabel("Predicted"); axes[0].set_ylabel("Residual (Actual - Predicted)")

axes[1].scatter(pred_B_test, resid_B, alpha=0.5, color="seagreen")
axes[1].axhline(0, color="black", linestyle="--")
axes[1].set_title("Model B Residuals", fontweight="bold")
axes[1].set_xlabel("Predicted"); axes[1].set_ylabel("Residual (Actual - Predicted)")
plt.suptitle("Residual Analysis: checking for systematic error patterns", fontsize=15, fontweight="bold")
plt.tight_layout()
plt.savefig("07_residuals.png", dpi=300)
plt.show()


# 6.3 - Error distribution comparison
fig, ax = plt.subplots(figsize=(9, 5.5))
sns.kdeplot(resid_A, color="crimson", fill=True, alpha=0.3, label="Model A residuals", ax=ax)
sns.kdeplot(resid_B, color="seagreen", fill=True, alpha=0.3, label="Model B residuals", ax=ax)
ax.axvline(0, color="black", linestyle="--", linewidth=1)
ax.set_title("Residual Distribution: Model A vs Model B", fontweight="bold")
ax.set_xlabel("Residual"); ax.legend()
plt.tight_layout()
plt.savefig("08_residual_distribution.png", dpi=300)
plt.show()


# 6.4 - R2 / RMSE bar comparison (train vs test, both models)
summary = pd.DataFrame({
    "Model": ["Model A (leaky)", "Model A (leaky)", "Model B (honest)", "Model B (honest)"],
    "Split": ["Train", "Test", "Train", "Test"],
    "R2": [mA_train["R2"], mA_test["R2"], mB_train["R2"], mB_test["R2"]],
    "RMSE": [mA_train["RMSE"], mA_test["RMSE"], mB_train["RMSE"], mB_test["RMSE"]]
})

fig, axes = plt.subplots(1, 2, figsize=(14, 5.5))
sns.barplot(data=summary, x="Split", y="R2", hue="Model", palette=["crimson", "seagreen"], ax=axes[0])
axes[0].set_title("R2 Score: Train vs Test", fontweight="bold")
axes[0].set_ylim(0, 1.05)
for container in axes[0].containers:
    axes[0].bar_label(container, fmt="%.3f")

sns.barplot(data=summary, x="Split", y="RMSE", hue="Model", palette=["crimson", "seagreen"], ax=axes[1])
axes[1].set_title("RMSE: Train vs Test", fontweight="bold")
for container in axes[1].containers:
    axes[1].bar_label(container, fmt="%.0f")
plt.suptitle("Model A vs Model B: Performance Summary", fontsize=15, fontweight="bold")
plt.tight_layout()
plt.savefig("09_metric_comparison_bars.png", dpi=300)
plt.show()
summary


# 6.5 - Feature coefficient comparison (standardized, Model B only)
scaler = StandardScaler()
X_B_scaled = scaler.fit_transform(X_B)
model_B_std = LinearRegression().fit(X_B_scaled, y_B)
coef_std = pd.DataFrame({"Feature": features_B, "Standardized Coeff": model_B_std.coef_}).sort_values(
    "Standardized Coeff", key=abs, ascending=True)

plt.figure(figsize=(9, 5.5))
colors = ["crimson" if v < 0 else "seagreen" for v in coef_std["Standardized Coeff"]]
plt.barh(coef_std["Feature"], coef_std["Standardized Coeff"], color=colors)
plt.axvline(0, color="black", linewidth=0.8)
plt.title("Model B: Standardized Feature Coefficients\n(comparable magnitudes, honest predictors only)", fontweight="bold")
plt.xlabel("Standardized coefficient")
plt.tight_layout()
plt.savefig("10_feature_coefficients.png", dpi=300)
plt.show()
